# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [29]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import seaborn as sns

### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [30]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']

In [31]:
df = pd.read_csv(url, names=names) 

# names=names se utiliza porque el archivo CSV no contiene encabezados. Con este parámetro se indica qué nombres debe asignar a las columnas.

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    768 non-null    int64  
 2   pres    768 non-null    int64  
 3   skin    768 non-null    int64  
 4   test    768 non-null    int64  
 5   mass    768 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   class   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [33]:
df.head()

,preg,plas,pres,skin,test,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [34]:
df.shape

(768, 9)

In [35]:
from sklearn.model_selection import train_test_split

# Variables predictoras
X = df.drop(columns=['class']).values

# Variable objetivo
y = df['class'].values

# Dividir los datos en entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [36]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(614, 8)
(154, 8)
(614,)
(154,)


### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [37]:
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import cross_val_score

In [38]:
seed = 42

In [39]:
tree = DecisionTreeClassifier(random_state=seed)

In [40]:
# Crear el BaggingClassifier
model = BaggingClassifier(
    estimator=tree,
    n_estimators=100,
    random_state=seed
)

# estimator=tree: utilizamos árboles de decisión como modelos base.
# n_estimators=100: creamos un conjunto de 100 árboles.
# random_state=seed: aseguramos la reproducibilidad.

In [41]:
# Aplicar validación cruzada con 10 folds
results = cross_val_score(
    model,
    X,
    y,
    cv=10,
    scoring='accuracy'
)

In [42]:
print("Accuracy de cada fold:", results)
print("Accuracy media:", results.mean())
print("Desviación estándar:", results.std())

#Accuracy: proporción de predicciones correctas.
#Accuracy media: rendimiento promedio del modelo.
# Desviación estándar: cuánto varía el rendimiento entre los 10 folds.

Accuracy de cada fold: [0.7012987  0.77922078 0.79220779 0.66233766 0.76623377 0.79220779
 0.83116883 0.85714286 0.72368421 0.80263158]
Accuracy media: 0.7708133971291866
Desviación estándar: 0.056511491183409916


### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [43]:
from sklearn.ensemble import RandomForestClassifier

In [44]:
# crear modelo

model_rf = RandomForestClassifier(
    n_estimators=100,
    max_features=3,
    random_state=seed
)

In [45]:
# Aplicar validación cruzada

results_rf = cross_val_score(
    model_rf,
    X,
    y,
    cv=10,
    scoring='accuracy'
)

In [46]:
print("Accuracy de cada fold:", results_rf)
print("Accuracy media:", results_rf.mean())
print("Desviación estándar:", results_rf.std())

Accuracy de cada fold: [0.72727273 0.79220779 0.77922078 0.66233766 0.72727273 0.77922078
 0.83116883 0.87012987 0.71052632 0.80263158]
Accuracy media: 0.768198906356801
Desviación estándar: 0.05851731937452667


### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [47]:
from sklearn.ensemble import AdaBoostClassifier

In [48]:
model_ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=30,
    random_state=seed
)

In [49]:
results_ada = cross_val_score(
    model_ada,
    X,
    y,
    cv=10,
    scoring='accuracy'
)

In [50]:
print("Accuracy de cada fold:", results_ada)
print("Accuracy media:", results_ada.mean())
print("Desviación estándar:", results_ada.std())

Accuracy de cada fold: [0.71428571 0.76623377 0.77922078 0.63636364 0.7012987  0.77922078
 0.79220779 0.79220779 0.72368421 0.82894737]
Accuracy media: 0.7513670539986329
Desviación estándar: 0.05385286224374289


In [51]:
# comparar los tres modelos

comparacion = pd.DataFrame({
    'Modelo': ['Bagging', 'Random Forest', 'AdaBoost'],
    'Accuracy media': [
        results.mean(),
        results_rf.mean(),
        results_ada.mean()
    ],
    'Desviación estándar': [
        results.std(),
        results_rf.std(),
        results_ada.std()
    ]
})

comparacion

,Modelo,Accuracy media,Desviación estándar
0,Bagging,0.770813,0.056511
1,Random Forest,0.768199,0.058517
2,AdaBoost,0.751367,0.053853


### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [28]:
from sklearn.ensemble import GradientBoostingClassifier

In [52]:
model_gb = GradientBoostingClassifier(
    n_estimators=100,
    random_state=seed
)

In [53]:
results_gb = cross_val_score(
    model_gb,
    X,
    y,
    cv=10,
    scoring='accuracy'
)

In [54]:
print("Accuracy de cada fold:", results_gb)
print("Accuracy media:", results_gb.mean())
print("Desviación estándar:", results_gb.std())

Accuracy de cada fold: [0.74025974 0.80519481 0.79220779 0.63636364 0.74025974 0.77922078
 0.79220779 0.80519481 0.71052632 0.81578947]
Accuracy media: 0.7617224880382775
Desviación estándar: 0.0530330182246875


In [55]:
comparacion = pd.DataFrame({
    'Modelo': ['Bagging', 'Random Forest', 'AdaBoost', 'Gradient Boosting'],
    'Accuracy media': [
        results.mean(),
        results_rf.mean(),
        results_ada.mean(),
        results_gb.mean()
    ],
    'Desviación estándar': [
        results.std(),
        results_rf.std(),
        results_ada.std(),
        results_gb.std()
    ]
})

comparacion.sort_values(by='Accuracy media', ascending=False)

,Modelo,Accuracy media,Desviación estándar
0,Bagging,0.770813,0.056511
1,Random Forest,0.768199,0.058517
3,Gradient Boosting,0.761722,0.053033
2,AdaBoost,0.751367,0.053853


Conclusiones: Ensemble Learning

Se han comparado cuatro modelos de clasificación (Bagging, Random Forest, AdaBoost y Gradient Boosting), evaluados mediante validación cruzada de 10 folds.

Bagging ha obtenido el mejor rendimiento, con una accuracy media del 77,08 %, seguido de Random Forest (76,82 %), Gradient Boosting (76,17 %) y AdaBoost (75,14 %).

Las diferencias de rendimiento son relativamente pequeñas, especialmente entre Bagging y Random Forest. Por su parte, Gradient Boosting presenta la menor desviación estándar, lo que indica una menor variabilidad entre las particiones.

En conclusión, Bagging es el modelo con mayor accuracy media en este ejercicio, aunque no podemos afirmar que sea significativamente superior sin realizar pruebas adicionales.

Dado que el objetivo es predecir la presencia de diabetes, sería recomendable complementar la evaluación con métricas como recall, precision y F1-score, especialmente para analizar la capacidad de detectar correctamente los casos positivos.

ara este ejercicio, Bagging, porque ha obtenido la mayor accuracy media.

Sin embargo, hay un aspecto importante: los cuatro modelos tienen resultados bastante similares. Por tanto, no podemos concluir que uno sea claramente mejor que los demás únicamente con estos resultados.

Además, al tratarse de un problema de detección de diabetes, sería importante comprobar los falsos negativos, es decir, las personas que tienen diabetes pero el modelo clasifica como no diabéticas.

Has completado correctamente los cuatro apartados de Ensemble Learning. Ya tienes sus modelos, las evaluaciones con validación cruzada y una conclusión comparativa para tu notebook.

### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [56]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [57]:
from xgboost import XGBClassifier

In [58]:
model_xgb = XGBClassifier(
    n_estimators=100,
    random_state=seed,
    eval_metric='logloss'
)

In [59]:
results_xgb = cross_val_score(
    model_xgb,
    X,
    y,
    cv=10,
    scoring='accuracy'
)

In [60]:
print("Accuracy de cada fold:", results_xgb)
print("Accuracy media:", results_xgb.mean())
print("Desviación estándar:", results_xgb.std())

Accuracy de cada fold: [0.68831169 0.74025974 0.75324675 0.67532468 0.74025974 0.75324675
 0.75324675 0.77922078 0.67105263 0.76315789]
Accuracy media: 0.7317327409432673
Desviación estándar: 0.036777663101667746


### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [62]:
comparacion = pd.DataFrame({
    'Modelo': [
        'Bagging',
        'Random Forest',
        'AdaBoost',
        'Gradient Boosting',
        'XGBoost'
    ],
    'Accuracy media': [
        results.mean(),
        results_rf.mean(),
        results_ada.mean(),
        results_gb.mean(),
        results_xgb.mean()
    ],
    'Desviación estándar': [
        results.std(),
        results_rf.std(),
        results_ada.std(),
        results_gb.std(),
        results_xgb.std()
    ]
})

In [ ]:
comparacion = comparacion.sort_values(
    by='Accuracy media',
    ascending=False
).reset_index(drop=True)

comparacion


# by='Accuracy media': ordena según la columna de accuracy.
# ascending=False: ordena de mayor a menor.
# reset_index(drop=True): reorganiza los índices del 0 al 4.

,Modelo,Accuracy media,Desviación estándar
0,Bagging,0.770813,0.056511
1,Random Forest,0.768199,0.058517
2,Gradient Boosting,0.761722,0.053033
3,AdaBoost,0.751367,0.053853
4,XGBoost,0.731733,0.036778


In [65]:
%pip install jinja2

   ---------------------------------------- 0.0/134.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/134.9 kB ? eta -:--:--
   --- ------------------------------------ 10.2/134.9 kB ? eta -:--:--
   -------- ------------------------------ 30.7/134.9 kB 435.7 kB/s eta 0:00:01
   -------------------- ------------------ 71.7/134.9 kB 653.6 kB/s eta 0:00:01
   -------------------------------------- 134.9/134.9 kB 997.3 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [66]:
comparacion.style.format({
    'Accuracy media': '{:.2%}',
    'Desviación estándar': '{:.2%}'
})

,Modelo,Accuracy media,Desviación estándar
0,Bagging,77.08%,5.65%
1,Random Forest,76.82%,5.85%
2,Gradient Boosting,76.17%,5.30%
3,AdaBoost,75.14%,5.39%
4,XGBoost,73.17%,3.68%


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

In [67]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['class']).values
y = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [68]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [69]:
param_grid_bag = {
    'n_estimators': [50, 100, 150],
    'estimator__max_depth': [3, 5, 10, None]
}

In [70]:
model_bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    random_state=42
)

grid_bag = GridSearchCV(
    estimator=model_bag,
    param_grid=param_grid_bag,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [71]:
grid_bag.fit(X_train, y_train)

print("Mejores parámetros:", grid_bag.best_params_)
print("Mejor accuracy CV:", grid_bag.best_score_)

Mejores parámetros: {'estimator__max_depth': 5, 'n_estimators': 100}
Mejor accuracy CV: 0.7672131147540984


In [72]:
y_pred_bag = grid_bag.predict(X_test)

accuracy_bag = accuracy_score(y_test, y_pred_bag)

print("Accuracy en test:", accuracy_bag)

Accuracy en test: 0.7662337662337663


In [73]:
param_grid_rf = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 10, None],
    'max_features': [2, 3, 4]
}

model_rf = RandomForestClassifier(random_state=42)

grid_rf = GridSearchCV(
    estimator=model_rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Mejores parámetros:", grid_rf.best_params_)
print("Mejor accuracy CV:", grid_rf.best_score_)

y_pred_rf = grid_rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Accuracy en test:", accuracy_rf)

Mejores parámetros: {'max_depth': None, 'max_features': 3, 'n_estimators': 100}
Mejor accuracy CV: 0.776915900306544
Accuracy en test: 0.7597402597402597


In [74]:
param_grid_ada = {
    'n_estimators': [30, 50, 100],
    'learning_rate': [0.01, 0.1, 1.0]
}

model_ada = AdaBoostClassifier(
    random_state=42
)

grid_ada = GridSearchCV(
    estimator=model_ada,
    param_grid=param_grid_ada,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_ada.fit(X_train, y_train)

print("Mejores parámetros:", grid_ada.best_params_)
print("Mejor accuracy CV:", grid_ada.best_score_)

y_pred_ada = grid_ada.predict(X_test)
accuracy_ada = accuracy_score(y_test, y_pred_ada)

print("Accuracy en test:", accuracy_ada)

Mejores parámetros: {'learning_rate': 1.0, 'n_estimators': 30}
Mejor accuracy CV: 0.767093162734906
Accuracy en test: 0.7857142857142857


In [75]:
param_grid_gb = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1],
    'max_depth': [2, 3, 5]
}

model_gb = GradientBoostingClassifier(
    random_state=42
)

grid_gb = GridSearchCV(
    estimator=model_gb,
    param_grid=param_grid_gb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)

print("Mejores parámetros:", grid_gb.best_params_)
print("Mejor accuracy CV:", grid_gb.best_score_)

y_pred_gb = grid_gb.predict(X_test)
accuracy_gb = accuracy_score(y_test, y_pred_gb)

print("Accuracy en test:", accuracy_gb)

Mejores parámetros: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 100}
Mejor accuracy CV: 0.7622684259629482
Accuracy en test: 0.7337662337662337


In [76]:
param_grid_xgb = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5]
}

model_xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_jobs=1
)

grid_xgb = GridSearchCV(
    estimator=model_xgb,
    param_grid=param_grid_xgb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

print("Mejores parámetros:", grid_xgb.best_params_)
print("Mejor accuracy CV:", grid_xgb.best_score_)

y_pred_xgb = grid_xgb.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print("Accuracy en test:", accuracy_xgb)

Mejores parámetros: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 150}
Mejor accuracy CV: 0.7574303611888579
Accuracy en test: 0.7337662337662337


In [77]:
comparacion_grid = pd.DataFrame({
    'Modelo': [
        'Bagging',
        'Random Forest',
        'AdaBoost',
        'Gradient Boosting',
        'XGBoost'
    ],
    'Accuracy CV': [
        grid_bag.best_score_,
        grid_rf.best_score_,
        grid_ada.best_score_,
        grid_gb.best_score_,
        grid_xgb.best_score_
    ],
    'Accuracy Test': [
        accuracy_bag,
        accuracy_rf,
        accuracy_ada,
        accuracy_gb,
        accuracy_xgb
    ]
})

comparacion_grid = comparacion_grid.sort_values(
    by='Accuracy Test',
    ascending=False
).reset_index(drop=True)

comparacion_grid

,Modelo,Accuracy CV,Accuracy Test
0,AdaBoost,0.767093,0.785714
1,Bagging,0.767213,0.766234
2,Random Forest,0.776916,0.759740
3,Gradient Boosting,0.762268,0.733766
4,XGBoost,0.757430,0.733766


### 10. Conclusiones finales